In [ ]:
from VesselProcessAPI import FreightowerAPI,Present,Tool
import pandas as pd

csv = Tool().read_initial_vessel_sheet(
    '巴西船表测试.csv',
    'departure',
    '2026-07-01',
    '2026-10-30'
).sort_index()
api = FreightowerAPI()
result = api.get_info_by_vessel_name(
    csv['query_vessel_name'],loading_country=['BR']
)
for s in ("unique", "multiple", "not_found"):
    print(s, sum(v["status"] == s for v in result.values()))

df = Present().get_info_by_vessel_name
df[['status','query_vessel_name','mmsi','shipType','flagName']].to_csv('mmsi.csv',index=False, encoding='utf-8-sig')

[歧义] IONIAN SEA
[歧义] IONIAN SEA
[模糊匹配] GALLIA GRAECA -> GALLIA GRAECA II
[模糊匹配] EGRET OSASIS -> EGRET OASIS


unique 223
multiple 1
not_found 1
读取存储，时间戳：2026-09-16 09:35:05


In [2]:
import pandas as pd
mmsi = pd.read_csv('mmsi.csv')['mmsi']
detail = api.set_mmsi(mmsi).get_ais().get_current_port().get_history_port().get_history_route(7).get_future_route().present.returns_by_mmsi

In [ ]:
from VesselProcessAPI import MatchTicket, Present
from pathlib import Path
import json, sys

# 预计算配票并缓存
mt = MatchTicket()
mt.generate_matched_tickets_with_port_calls('巴西船表测试.csv', 'departure', 'CN')
cache_path = Path(sys.modules[MatchTicket.__module__].__file__).with_name(".json") / ".matched_tickets_with_port_calls.json"
cache = json.loads(cache_path.read_text(encoding="utf-8"))
print(f"缓存更新时间: {cache.get('updated_at')}")

# 只消费缓存，做窗口过滤
df = Present().combine_initial_vessel_sheet_with_query_result(
    '2026-09-01',
    '2026-09-30',
)
display(df.columns)
df[['status','query_vessel_name','arrival_time', 'departure_time',
       'call_type','port_name_cn','port_name_en','quantity']].to_csv('船表.csv',index=False, encoding='utf-8-sig')